# Дообучение Laya под этапы review_fake и ai_text

Laya — энкодер типизированных решений: отвечает «да/нет» одним проходом вместо
генерации. Без дообучения на наших текстах она отвечает **по позиции варианта**,
а не по смыслу (AUROC 0.93 прямо и 0.07 наоборот, `docs/laya.md`). Этот ноутбук
чинит ровно это.

Что нужно заранее: в разделе «Laya» интерфейса нажать «Собрать датасет для Laya»
и забрать из `data/train/laya/` два файла — `train.jsonl` и `test.jsonl`.

Среда: **Runtime → Change runtime type → T4 GPU**. Одной карты хватает,
официальный ноутбук Laya требует двух только из-за DDP. Час на нескольких сотнях
примеров.

> **Данные личные.** В отзывах встречаются имена сотрудников. Ноутбук держите
> приватным, датасет в Drive общего доступа не кладите, обученную модель в
> Hugging Face Hub не публикуйте.

## 1. Проверяем карту

In [ ]:
# Без GPU дальше идти незачем: на CPU полный прогон падает по памяти.
import torch

assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0), '·', torch.__version__)
!nvidia-smi --query-gpu=memory.total,memory.free --format=csv

## 2. Ставим Laya

Пакет называется `laya`: из него берутся `build_model`, `build_sequence` и
правило оценки, на которых держится обучение. Версии `torch` из образа Colab
хватает, переустанавливать её не нужно.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir laya

In [ ]:
import laya

print('laya', laya.__version__)

## 3. Берём скрипт обучения

`training/laya_finetune.py` — однопроцессная адаптация официального ноутбука:
тот же цикл RLCD плюс кросс-энтропия, та же калибровка температур на отложенном
срезе, тот же формат сохранения. Репозиторий публичный, поэтому просто клон.

In [ ]:
from pathlib import Path

REPO = 'https://github.com/lineSence/FuckHR.git'
if not Path('FuckHR').exists():
    !git clone --depth 1 $REPO

SCRIPT = Path('FuckHR/training/laya_finetune.py')
assert SCRIPT.exists(), 'скрипт не найден: проверьте клон или загрузите файл вручную'
print(SCRIPT.read_text(encoding='utf-8').splitlines()[0])

## 4. Загружаем свой датасет

Кнопка ниже просит `train.jsonl` и `test.jsonl` (можно выбрать оба сразу).
В репозитории их нет и быть не должно: это выгрузка из личной базы.

In [ ]:
import json
from collections import Counter
from pathlib import Path

DATA = Path('laya-data')
DATA.mkdir(exist_ok=True)

if not (DATA / 'train.jsonl').exists():
    from google.colab import files
    for name, blob in files.upload().items():
        (DATA / name).write_bytes(blob)

def summary(path):
    rows = [json.loads(line) for line in path.open(encoding='utf-8') if line.strip()]
    stages = Counter(r['workflow'] for r in rows)
    yes = Counter(
        r['workflow'] for r in rows
        if json.loads(r['gold'])['verdict']['label'] == 'A'
    )
    return rows, stages, yes

for name in ('train.jsonl', 'test.jsonl'):
    rows, stages, yes = summary(DATA / name)
    print(name, '·', len(rows), 'строк')
    for stage, total in stages.items():
        print('   {:<12} {:>4} строк, из них «да» {}'.format(stage, total, yes[stage]))

# Меньше 50 положительных на этап — замер на отложенной части ничего не покажет.
_, _, yes = summary(DATA / 'train.jsonl')
for stage, count in yes.items():
    if count < 50:
        print('мало положительных на {}: {} < 50, копите разметку дальше'.format(stage, count))

## 5. Учим

Гиперпараметры — из официального ноутбука; менять их стоит только если видите,
зачем. `--max-steps 20` в первом запуске полезен: проверяет, что всё сходится,
за пару минут вместо часа.

In [ ]:
OUT = 'laya-fuckhr'
BASE = 'convaiinnovations/laya-multilingual'  # мультиязычный обязателен: тексты русские

!python FuckHR/training/laya_finetune.py \
    --data laya-data --base $BASE --out $OUT \
    --epochs 4 --batch 8 --accum 8 \
    --lr-encoder 2.5e-5 --lr-head 1e-4

## 6. Читаем замер

Критерий приёма один: **AUROC не ниже 0.7 при обоих порядках вариантов**.
Высокий прямой при низком перевёрнутом означает, что модель по-прежнему
отвечает по позиции, и включать её нельзя.

In [ ]:
import json
from pathlib import Path

report = json.loads((Path(OUT) / 'eval.json').read_text(encoding='utf-8'))
ok = True
for stage, part in report.items():
    прямой = part.get('auroc_прямой', 0)
    обратный = part.get('auroc_перевёрнутый', 0)
    ok = ok and min(прямой, обратный) >= 0.7
    print('{:<12} вопросов {:>4} · точность {:.2f} · AUROC {:.2f} / {:.2f}'.format(
        stage, part.get('вопросов', 0), part.get('точность', 0), прямой, обратный))

print('\nгодится' if ok else '\nне годится: нужен либо ещё датасет, либо другой чекпойнт')

## 7. Забираем модель

In [ ]:
!zip -qr laya-fuckhr.zip $OUT
!du -h laya-fuckhr.zip

from google.colab import files
files.download('laya-fuckhr.zip')

## 8. Что сделать на своей машине

1. Распакуйте архив рядом с проектом, например в `models/laya-fuckhr`.
2. В настройках, группа «Гейты и разметка», укажите путь в «Чекпойнт решателя»
   (`LAYA_MODEL`). Замер из `eval.json` появится в разделе «Laya».
3. Там же нажмите «Сравнить с текущей моделью»: прогон возьмёт отложенную
   выборку и покажет, что решатель даёт по сравнению с обычной моделью.
4. Только после этого включайте `LAYA_ENABLED` — и гейтом, а не заменой:
   уверенное «да» или «нет» экономит вызов, середина и расхождение порядков
   по-прежнему идут через шлюз.

Подробности и командный путь — `docs/laya-finetune.md`.